In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/dayanithaam/dataset3/AgroQA Dataset.csv
/kaggle/input/datasets/dayanithaam/kcc-agri/kcc_cleaned.csv
/kaggle/input/datasets/dayanithaam/kcc-extended/kcc_extended.csv
/kaggle/input/datasets/dayanithaam/nlp-agri-dataset/data.csv
/kaggle/input/datasets/daskoushik/farmers-call-query-data-qa/questionsv4.csv


In [2]:
import pandas as pd

# Load the dataset
# If it's a CSV, use pd.read_csv. If it's Excel, use pd.read_excel
df = pd.read_csv('/kaggle/input/datasets/dayanithaam/nlp-agri-dataset/data.csv') #initial kcc

# Display the first few rows to verify the structure
df.head()



,question,answer,intent,state,crop
0,paddy (dhan) nutrient management in paddy crop-,recommendations of chamatkar(12% chelated zinc...,nutrient management,telangana,paddy (dhan)
1,papaya sucking pest in papaya management,recommended to spray fipronil +imidacloprid (p...,plant protection,telangana,papaya
2,sorghum (jowar/great millet) stem borer manage...,recommended to spray cartap hydrochloride (pad...,plant protection,telangana,sorghum (jowar/great millet)
3,cowpea (vegetable) management of mosaic diseas...,suggested to remove and destroy severely affec...,plant protection,kerala,cowpea (vegetable)
4,arecanut management of basal rotting in arecanut,suggested to soil drench hexaconazole 1ml per ...,plant protection,kerala,arecanut


In [3]:
# Get the number of unique crops
unique_crops_count = df['crop'].nunique()

# Get the list of unique crop names
unique_crops_list = df['crop'].unique()

print(f"Total number of unique crops: {unique_crops_count}")


Total number of unique crops: 236


In [4]:
# Get the count for each unique crop
crop_counts = df['crop'].value_counts()

print("Count of each crop:")
print(crop_counts)

# If you want to see the percentage distribution
crop_percentages = df['crop'].value_counts(normalize=True) * 100
print("\nPercentage distribution of crops:")
print(crop_percentages)

Count of each crop:
crop
others            3897
paddy (dhan)      2911
chillies          1709
coconut            870
banana             846
                  ... 
7169                 1
african sarson       1
barseem              1
poplar               1
guinea grass         1
Name: count, Length: 236, dtype: int64

Percentage distribution of crops:
crop
others            16.310216
paddy (dhan)      12.183485
chillies           7.152723
coconut            3.641234
banana             3.540786
                    ...    
7169               0.004185
african sarson     0.004185
barseem            0.004185
poplar             0.004185
guinea grass       0.004185
Name: proportion, Length: 236, dtype: float64


In [5]:
# Count how many intents contain the word "crop" (case-insensitive)
crop_intent_count = df['intent'].str.contains('crop', case=False, na=False).sum()

print(f"Number of intents that mention 'crop': {crop_intent_count}")

# If you want to see the specific unique intents that mention "crop"
unique_crop_intents = df[df['intent'].str.contains('crop', case=False, na=False)]['intent'].unique()

print("\nUnique intent names that contain 'crop':")
print(unique_crop_intents)
# Show the top 20 intents
print(df['intent'].value_counts().head(20))

#remove government schemes and market information
# Define the intents you want to remove
intents_to_drop = ['market information', 'government schemes']

# Keep only the rows where the intent is NOT in that list
df = df[~df['intent'].isin(intents_to_drop)]

# Reset the index since we've removed rows
df = df.reset_index(drop=True)

print(f"Rows remaining after filtering: {len(df)}")


Number of intents that mention 'crop': 35

Unique intent names that contain 'crop':
['crop insurance' 'spices and condiment crops']
intent
plant protection                                                                            9310
nutrient management                                                                         3953
cultural practices                                                                          2600
government schemes                                                                          1644
market information                                                                          1379
fertilizer use and availability                                                              800
weed management                                                                              582
sowing time and weather                                                                      578
seeds                                                                                

In [6]:
import pandas as pd 
df2 = pd.read_csv('/kaggle/input/datasets/daskoushik/farmers-call-query-data-qa/questionsv4.csv') #90kdata

# 1. Basic Cleaning: Remove leading/trailing whitespace
df2['questions'] = df2['questions'].str.strip()
df2['answers'] = df2['answers'].str.strip()

# 2. Case Normalization: Convert to lowercase for consistency
df2['questions'] = df2['questions'].str.lower()
df2['answers'] = df2['answers'].str.lower()


# 2. Rename columns to match your first dataset (singular)
df2 = df2.rename(columns={'questions': 'question', 'answers': 'answer'})

df2 = df2.dropna(subset=['question', 'answer'])
df2 = df2.drop_duplicates(subset=['question'])
# 4. Remove "asking about" from the start of the strings
# The '^' symbol ensures it only removes from the START, not the middle
df2['question'] = df2['question'].str.replace(r'^asking about\s*', '', regex=True)
df2['answer'] = df2['answer'].str.replace(r'^suggested to\s*', '', regex=True)
df2['question'] = df2['question'].str.replace(r'^query regarding\s*', '', regex=True)
df2['answer'] = df2['answer'].str.replace(r'^suggested him\s*', '', regex=True)
df2['question'] = df2['question'].str.replace(r'^quary related\s*', '', regex=True)
df2['answer'] = df2['answer'].str.replace(r'^adviced him\s*', '', regex=True)
df2['answer'] = df2['answer'].str.replace(r'^adviced to\s*', '', regex=True)

# 5. Final strip to clean up any remaining edge spaces
df2['question'] = df2['question'].str.strip()

# Check the results
print(df2.head())


                                            question  \
0  the control measure for aphid infestation in m...   
1  the control measure of flower drop problem in ...   
2  how to avail kisan credit card loan for sali c...   
3                   source of early ahu rice variety   
4  asking that he has not got proper friut from h...   

                                              answer  
0            to spray rogor@2ml/lit.at evening time.  
1  to apply fertilizer in recommended dose like u...  
2  consult with officer-marketing and recovery (r...  
3  take early ahu rice variety from atic,jorhat,a...  
4  to aplly recommended fertilizer dose-urea@1.5 ...  


In [7]:
import pandas as pd
df3= pd.read_csv('/kaggle/input/datasets/dayanithaam/dataset3/AgroQA Dataset.csv') #3k
df3 = df3.rename(columns={
    'Question': 'question',
    'Answer': 'answer',
    'Crop': 'crop'
})

# 2. Add missing columns with 'unknown' or NaN to stay consistent with df1
df3['intent'] = 'unknown'
df3['state'] = 'unknown'
column_order = ['question', 'answer', 'intent', 'state', 'crop']
df3 = df3[column_order]

print("Updated 3rd Dataset Structure:")
print(df3.head())


Updated 3rd Dataset Structure:
                                            question  \
0  Apart from hand weeding, what other method use...   
1  Apart from insecticide, what other method used...   
2  Apart from sun drying which other method used ...   
3  Apart from sun drying, what other method can I...   
4           As a farmer when should I harvest beans.   

                                              answer   intent    state  \
0                    Machinery weeders are available  unknown  unknown   
1  Use resistant verities and increase on water a...  unknown  unknown   
2    Use tarpaulins or cemented floor free from dust  unknown  unknown   
3                                       Solar driers  unknown  unknown   
4  When the beans pods are yellowish green or dry...  unknown  unknown   

      crop  
0    maize  
1    beans  
2    maize  
3  cassava  
4    beans  


In [8]:
import pandas as pd
df4= pd.read_csv('/kaggle/input/datasets/dayanithaam/kcc-extended/kcc_extended.csv')
intents_to_drop = ['market information', 'government schemes']

# Keep only the rows where the intent is NOT in that list
df4 = df4[~df4['intent'].isin(intents_to_drop)]

df4['crop'] = 'unknown'

# 3. Ensure column names are lowercase (to avoid 'Question' vs 'question' issues)
df4.columns = [col.lower().strip() for col in df4.columns]

df4.head()


,question,answer,intent,state,crop
0,bengal gram (gram/chick pea/kabuli/chana) plan...,recommended spray chlorpyriphos - 20 ec. @ 2 m...,plant protection,karnataka,unknown
2,chillies chilli thrips and fruit borer managem...,recommended to spray broflanilide 300g/l sc 34...,plant protection,telangana,unknown
3,others bio fertilizer availability ?,dr. m. sreerekha principal scientist (agro) an...,bio-pesticides and bio-fertilizers,telangana,unknown
4,drum stick disease management in drum stick,recommneded to spray 13-0-45 1 kg + agro mini ...,plant protection,telangana,unknown
5,sesame (gingelly/til)/sesamum crop duration of...,crop duration of sesamum : 80-85 days crop,plant protection,telangana,unknown


In [9]:
import pandas as pd

df5= pd.read_csv('/kaggle/input/datasets/dayanithaam/kcc-agri/kcc_cleaned.csv')
master_columns = ['question', 'answer', 'intent', 'state', 'crop']

# 3. Reorder the columns
# This 'slices' the dataframe into the correct sequence
df5 = df5[master_columns]

df5.head()


,question,answer,intent,state,crop
0,fine paddy varieties information,kunnaram rice knm 733 fine variety telangana v...,seeds,TELANGANA,paddy
1,turmeric weed management,spray propaquizafop 250 ml 200 litre of water ...,cultural practices,TELANGANA,turmeric
2,varieties of green gram,average crop yield 5 6 quintal acre.,varieties,TELANGANA,green gram
3,weed management in bitter gourd,recommended hand weeding in bittergourd.,cultural practices,TELANGANA,bitter gourd
4,query weed management in cotton,do weeding with help of labour .,weed management,TELANGANA,cotton


In [10]:
# 1. Add the missing columns with 'unknown' or NaN
# This ensures that when we merge, these rows don't cause column misalignment
df2['intent'] = 'unknown'
df2['state'] = 'unknown'

# Note: Earlier we created a 'crop' column for df2 using keyword extraction. 
# If you haven't done that yet, we can initialize it as 'unknown' for now:
if 'crop' not in df2.columns:
    df2['crop'] = 'unknown'

# 2. Define the Master Column Order
# (Based on your first dataset's structure)
master_columns = ['question', 'answer', 'intent', 'state', 'crop']

# 3. Reorder the columns
# This 'slices' the dataframe into the correct sequence
df2 = df2[master_columns]

# Get all unique crops from the dataset that actually has labels
known_crops = df5['crop'].unique().tolist()

# Remove 'unknown' if it's in there
if 'unknown' in known_crops:
    known_crops.remove('unknown')
known_crops = [str(c).lower() for c in known_crops if pd.notna(c)]

print(f"Searching for these {len(known_crops)}")

def extract_crop_name(question):
    
    question_str = str(question).lower()
    
    for crop in known_crops:
        if crop in question_str:
            return crop
    return 'unknown'

# Re-run the apply
df2['crop'] = df2['question'].apply(extract_crop_name)

print(df2['crop'].value_counts())

# Calculate the success rate
converted = len(df2[df2['crop'] != 'unknown'])
print(f"\nSuccessfully identified crops for {converted} out of 90,000 rows.")
# 4. Final verification
print("2nd Dataset - Standardized Columns:")
print(df2.columns.tolist())
print("\nFirst 5 rows:")
df2.head()

Searching for these 279
crop
unknown         38101
chilli           3730
paddy            3502
potato           2704
brinjal          2507
                ...  
apricot             1
horse gram          1
brown sarson        1
triticale           1
eculeptous          1
Name: count, Length: 190, dtype: int64

Successfully identified crops for 50767 out of 90,000 rows.
2nd Dataset - Standardized Columns:
['question', 'answer', 'intent', 'state', 'crop']

First 5 rows:


,question,answer,intent,state,crop
0,the control measure for aphid infestation in m...,to spray rogor@2ml/lit.at evening time.,unknown,unknown,mustard
1,the control measure of flower drop problem in ...,to apply fertilizer in recommended dose like u...,unknown,unknown,coconut
2,how to avail kisan credit card loan for sali c...,consult with officer-marketing and recovery (r...,unknown,unknown,unknown
3,source of early ahu rice variety,"take early ahu rice variety from atic,jorhat,a...",unknown,unknown,unknown
4,asking that he has not got proper friut from h...,to aplly recommended fertilizer dose-urea@1.5 ...,unknown,unknown,coconut


In [11]:
import pandas as pd

# List of all your dataframes
all_dfs = [df, df2, df3, df4]

# Merge into one master dataframe
# ignore_index=True gives you a fresh index from 0 to the total number of rows
final_df = pd.concat(all_dfs, axis=0, ignore_index=True)

# Remove any rows that are identical across all columns
# This prevents skewed visualizations from redundant data
final_df = final_df.drop_duplicates()

print(f"Merge Complete!")
print(f"Total records in the combined dataset: {len(final_df)}")

# Save the final dataset to a CSV file
# index=False prevents Pandas from adding a new column for the row numbers
final_df.to_csv('final_agricultural_dataset.csv', index=False)

print("File saved successfully as 'final_agricultural_dataset.csv'")

Merge Complete!
Total records in the combined dataset: 129796
File saved successfully as 'final_agricultural_dataset.csv'


In [12]:
# 1. Define the specific list of crops you are looking for
# We use lowercase because we standardized the crop column earlier
target_animals = ['goat']

# 2. Filter the dataframe
# .isin() checks if the value in the 'crop' column matches any item in our list
animal_crop_df = df2[df2['crop'].isin(target_animals)]

# 3. Get the total count
total_count = len(animal_crop_df)

print(f"Total rows where crop is 'pig', 'duck', or 'flying duck': {total_count}")

# 4. Break down the count per specific type
print("\nBreakdown per animal type:")
print(animal_crop_df.head())

Total rows where crop is 'pig', 'duck', or 'flying duck': 0

Breakdown per animal type:
Empty DataFrame
Columns: [question, answer, intent, state, crop]
Index: []


In [13]:
import pandas as pd

# List of all your dataframes
all_dfs = [df5, df2, df3]

# Merge into one master dataframe
# ignore_index=True gives you a fresh index from 0 to the total number of rows
final_df = pd.concat(all_dfs, axis=0, ignore_index=True)

# 1. Define the list of crops to remove
to_remove = ['pig', 'duck', 'flying duck']

# 2. Use the ~ (NOT) operator to filter them out
# This keeps only rows where the crop is NOT pig, duck, or flying duck
final_df = final_df[~final_df['crop'].isin(to_remove)]

# 3. Reset the index
# Since we deleted rows, we reset the row numbers to stay sequential
final_df = final_df.reset_index(drop=True)

print(f"Removal complete. Current row count: {len(final_df)}")

# 4. Quick verification
print("\nCheck if any target rows remain:")
print(final_df['crop'].isin(to_remove).value_counts())

# Remove any rows that are identical across all columns
# This prevents skewed visualizations from redundant data
final_df = final_df.drop_duplicates()

print(f"Merge Complete!")
print(f"Total records in the combined dataset: {len(final_df)}")

# Save the final dataset to a CSV file
# index=False prevents Pandas from adding a new column for the row numbers
final_df.to_csv('data.csv', index=False)

print("File saved successfully as 'data.csv'")


Removal complete. Current row count: 143938

Check if any target rows remain:
crop
False    143938
Name: count, dtype: int64
Merge Complete!
Total records in the combined dataset: 143818
File saved successfully as 'data.csv'


In [14]:


to_remove = ['pig', 'duck', 'flying duck']
print(df2['crop'].isin(to_remove).value_counts())
# 2. Use the ~ (NOT) operator to filter them out
# This keeps only rows where the crop is NOT pig, duck, or flying duck
df2 = df2[~df2['crop'].isin(to_remove)]

print(f"Removal complete. Current row count: {len(df2)}")


df2 = df2.reset_index(drop=True)

# Remove any rows that are identical across all columns
# This prevents skewed visualizations from redundant data
df2 = df2.drop_duplicates()

print(f"Merge Complete!")
print(f"Total records in the combined dataset: {len(df2)}")

# Save the final dataset to a CSV file
# index=False prevents Pandas from adding a new column for the row numbers
df2.to_csv('df2.csv', index=False)

print("File saved successfully as 'df2.csv'")
df2.head()

crop
False    87062
True      1806
Name: count, dtype: int64
Removal complete. Current row count: 87062
Merge Complete!
Total records in the combined dataset: 86943
File saved successfully as 'df2.csv'


,question,answer,intent,state,crop
0,the control measure for aphid infestation in m...,to spray rogor@2ml/lit.at evening time.,unknown,unknown,mustard
1,the control measure of flower drop problem in ...,to apply fertilizer in recommended dose like u...,unknown,unknown,coconut
2,how to avail kisan credit card loan for sali c...,consult with officer-marketing and recovery (r...,unknown,unknown,unknown
3,source of early ahu rice variety,"take early ahu rice variety from atic,jorhat,a...",unknown,unknown,unknown
4,asking that he has not got proper friut from h...,to aplly recommended fertilizer dose-urea@1.5 ...,unknown,unknown,coconut


In [15]:
from datasets import load_dataset
from huggingface_hub import login
login("KEY")

ds = load_dataset("yahma/alpaca-cleaned")
print(ds)


README.md: 0.00B [00:00, ?B/s]

alpaca_data_cleaned.json:   0%|          | 0.00/44.3M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/51760 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['output', 'input', 'instruction'],
        num_rows: 51760
    })
})


In [16]:
# Convert the 'train' split to a Pandas DataFrame
hf_df = ds['train'].to_pandas()

# Select only the columns you want and rename them to match your final_df
# We take only the first 10,000 rows
hf_df = hf_df[['instruction', 'output']].head(10000)

# Rename to be consistent with existing data
hf_df = hf_df.rename(columns={'instruction': 'question', 'output': 'answer'})

# Add the placeholder columns
hf_df['intent'] = 'unknown'
hf_df['state'] = 'unknown'
hf_df['crop'] = 'unknown'

print(f"Extracted {len(hf_df)} rows from Hugging Face dataset.")

hf_df.head()

Extracted 10000 rows from Hugging Face dataset.


,question,answer,intent,state,crop
0,Give three tips for staying healthy.,1. Eat a balanced and nutritious diet: Make su...,unknown,unknown,unknown
1,What are the three primary colors?,"The three primary colors are red, blue, and ye...",unknown,unknown,unknown
2,Describe the structure of an atom.,An atom is the basic building block of all mat...,unknown,unknown,unknown
3,How can we reduce air pollution?,There are several ways to reduce air pollution...,unknown,unknown,unknown
4,Pretend you are a project manager of a constru...,I had to make a difficult decision when I was ...,unknown,unknown,unknown


In [17]:
all_dfs = [df5, df2, df3,hf_df]

# Merge into one master dataframe
# ignore_index=True gives you a fresh index from 0 to the total number of rows
final_df = pd.concat(all_dfs, axis=0, ignore_index=True)

# 1. Define the list of crops to remove
to_remove = ['pig', 'duck', 'flying duck']


# This keeps only rows where the crop is NOT pig, duck, or flying duck
final_df = final_df[~final_df['crop'].isin(to_remove)]

# 3. Reset the index
# Since we deleted rows, we reset the row numbers to stay sequential
final_df = final_df.reset_index(drop=True)

print(f"Removal complete. Current row count: {len(final_df)}")

# 4. Quick verification
print("\nCheck if any target rows remain:")
print(final_df['crop'].isin(to_remove).value_counts())

# Remove any rows that are identical across all columns

final_df = final_df.drop_duplicates()


print(f"Total records in the combined dataset: {len(final_df)}")

final_df.to_csv('datafinal.csv', index=False)

print("File saved successfully as 'datafinal.csv'")


Removal complete. Current row count: 153819

Check if any target rows remain:
crop
False    153819
Name: count, dtype: int64
Merge Complete!
Total records in the combined dataset: 153818
File saved successfully as 'datafinal.csv'


In [21]:
all_dfs = [df2,df3]

# Merge into one master dataframe
# ignore_index=True gives you a fresh index from 0 to the total number of rows
final_df = pd.concat(all_dfs, axis=0, ignore_index=True)

# 1. Define the list of crops to remove
to_remove = ['pig', 'duck', 'flying duck']

# 2. Use the ~ (NOT) operator to filter them out
# This keeps only rows where the crop is NOT pig, duck, or flying duck
final_df = final_df[~final_df['crop'].isin(to_remove)]

# 3. Reset the index
# Since we deleted rows, we reset the row numbers to stay sequential
final_df = final_df.reset_index(drop=True)

print(f"Removal complete. Current row count: {len(final_df)}")

# 4. Quick verification
print("\nCheck if any target rows remain:")
print(final_df['crop'].isin(to_remove).value_counts())

# Remove any rows that are identical across all columns
# This prevents skewed visualizations from redundant data
final_df = final_df.drop_duplicates()
print(f"Merge Complete!")
print(f"Total records in the combined dataset: {len(final_df)}")
final_df.head()
final_df.to_csv('generic.csv', index=False)
final_df.head()

print("File saved successfully as 'generic.csv'")



Removal complete. Current row count: 89987

Check if any target rows remain:
crop
False    89987
Name: count, dtype: int64
Merge Complete!
Total records in the combined dataset: 89986
File saved successfully as 'generic.csv'
